<a href="https://colab.research.google.com/github/KULDEEPSONI-source/TIMESERIES-ANALYSIS-FORCASTING/blob/main/MODELS_IN_TIME_SERIES_ANALYSIS_AND_FORCASTING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ARIMA (AutoRegressive Integrated Moving Average)

ARIMA is one of the most widely used classical statistical models for time series forecasting. It combines three key components into a unified framework.

---

### 1. Model Structure

ARIMA is denoted as **ARIMA(p, d, q)** where:

| Parameter | Meaning | Description |
|---|---|---|
| **p** | Autoregressive order | Number of lag observations included in the model |
| **d** | Differencing order | Number of times the data is differenced to make it stationary |
| **q** | Moving Average order | Size of the moving average window |

---

### 2. The Three Components

#### **AR(p) — Autoregressive**
The current value is a linear function of past values plus noise:

$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + ... + \phi_p y_{t-p} + \epsilon_t$$

- Uses past values to predict future values
- Captures momentum and mean reversion in the series

#### **I(d) — Integrated (Differencing)**
Transforms non-stationary data into stationary data:

$$\Delta y_t = y_t - y_{t-1}$$

- Removes trend by subtracting previous observation
- **d=0**: Original series (already stationary)
- **d=1**: First differences (common for trending data)
- **d=2**: Second differences (rarely needed)

#### **MA(q) — Moving Average**
The current value depends on past forecast errors:

$$y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + ... + \theta_q \epsilon_{t-q}$$

- Models shock/impulse effects that decay over time
- Captures short-term dependencies

---

### 3. Stationarity Requirement

ARIMA requires the time series to be **stationary** (constant mean, variance, autocorrelation over time).

| Test | Purpose | Interpretation |
|---|---|---|
| **ADF Test** (Augmented Dickey-Fuller) | Tests for unit root | p < 0.05 → stationary |
| **KPSS Test** | Tests for trend stationarity | p > 0.05 → stationary |
| **ACF/PACF Plots** | Identify AR/MA orders | Visual inspection |

**If non-stationary:** Apply differencing (increase `d`) until stationary.

---

### 4. How to Choose (p, d, q)

| Step | Action | Tool |
|---|---|---|
| 1. Check stationarity | ADF test, visualize series | `adfuller()` |
| 2. Determine `d` | Difference until stationary | ACF decay, ADF test |
| 3. Determine `p` | PACF plot — significant lags | `plot_pacf()` |
| 4. Determine `q` | ACF plot — significant lags | `plot_acf()` |
| 5. Validate | AIC/BIC criteria, residual analysis | `model.summary()` |

**Rules of thumb from ACF/PACF:**
- **AR(p)**: PACF cuts off after lag p, ACF tails off
- **MA(q)**: ACF cuts off after lag q, PACF tails off
- **ARMA(p,q)**: Both tail off

---

### 5. SARIMA — Seasonal Extension

For data with seasonality: **SARIMA(p,d,q)(P,D,Q)m**

| Parameter | Meaning |
|---|---|
| **P, D, Q** | Seasonal AR, differencing, MA orders |
| **m** | Seasonal period (e.g., 12 for monthly, 7 for daily) |

Example: `SARIMA(1,1,1)(1,1,1)12` — monthly data with yearly seasonality.

---

### 6. Practical Example (from earlier demo)

```python
from statsmodels.tsa.arima.model import ARIMA

# Fit ARIMA(2, 1, 2)
model = ARIMA(train_data, order=(2, 1, 2))
fitted = model.fit()

# Forecast next 365 days
forecast = fitted.forecast(steps=365)

# Model diagnostics
print(fitted.summary())
```

**Results from demo:**
- MAE: 11.31
- RMSE: 13.56
- MAPE: 5.79%

ARIMA underperformed Holt-Winters because the data had strong **multiplicative seasonality**, which ARIMA(2,1,2) without seasonal terms couldn't capture. A **SARIMA** model would likely perform better.

---

### 7. ARIMA vs. Other Models

| Aspect | ARIMA | Holt-Winters | ML (XGBoost) | Deep Learning |
|---|---|---|---|---|
| **Data size** | Small–medium | Small–medium | Medium–large | Large |
| **Seasonality** | Needs SARIMA | Built-in | Manual features | Automatic |
| **Trend** | Via differencing | Built-in | Manual features | Automatic |
| **Exogenous vars** | ARIMAX | No | Yes | Yes |
| **Interpretability** | High | High | Medium | Low |
| **Non-linearity** | No | No | Yes | Yes |
| **Training speed** | Fast | Fast | Medium | Slow |

---

### 8. Common Issues & Fixes

| Problem | Cause | Solution |
|---|---|---|
| Poor forecasts | Non-stationary data | Increase `d`, check ADF test |
| Overfitting | Too many parameters | Reduce p/q, use AIC/BIC |
| Missed seasonality | Using ARIMA not SARIMA | Add seasonal terms |
| Volatility clustering | Heteroscedasticity | Use GARCH-ARIMA hybrid |
| Long-term drift | Structural breaks | Segment data, use regime-switching models |

---

### 9. Auto-ARIMA

Automatically selects best (p,d,q) using information criteria:

```python
from pmdarima import auto_arima

model = auto_arima(
    train_data,
    seasonal=False,
    d=None,           # Let model determine differencing
    max_p=5, max_q=5,
    trace=True,
    error_action='ignore',
    suppress_warnings=True
)
```

Or using `statsforecast`:

```python
from statsforecast.models import AutoARIMA

model = AutoARIMA(season_length=7)  # Weekly seasonality
```

---

### 10. When to Use ARIMA

| Use ARIMA | Don't Use ARIMA |
|---|---|
| Small-to-medium datasets | Very large, high-dimensional data |
| Clear linear patterns | Strong non-linear relationships |
| Need interpretability | Need black-box accuracy |
| Stationary or easily differenced | Multiple complex seasonalities |
| Baseline comparison | Real-time streaming forecasts |

---

### Key Takeaway

ARIMA is a **foundational, interpretable** model best suited for simpler time series with linear trends and limited data. For complex, seasonal, or large-scale forecasting, consider **SARIMA**, **Holt-Winters**, **Prophet**, or modern deep learning approaches.

## ACF (Autocorrelation Function) & PACF (Partial Autocorrelation Function)

ACF and PACF are essential diagnostic tools for identifying the structure of a time series and selecting ARIMA model parameters. They are the starting point for any ARIMA modeling workflow.

---

## 1. Autocorrelation Function (ACF)

### Definition
ACF measures the **correlation between a time series and its lagged values** at different time intervals.

$$\rho_k = \frac{\text{Cov}(y_t, y_{t-k})}{\text{Var}(y_t)}$$

Where $k$ = lag, $\rho_k$ = autocorrelation at lag $k$

### What It Tells You
- How strongly the current value is related to past values
- The **duration of influence** — how long shocks persist in the series
- Whether the series is stationary (ACF should decay quickly for stationary data)

### ACF Patterns & Interpretation

| Pattern | Indicates | Action |
|---|---|---|
| **Slow, gradual decay** | Non-stationary (trend present) | Apply differencing (increase `d`) |
| **Sudden drop to zero** | Stationary series | `d = 0`, check PACF for AR order |
| **Oscillating decay** | Seasonal pattern | Consider seasonal differencing |
| **Significant spike at lag k** | MA(q) order = k | Cutoff point indicates `q` |

---

## 2. Partial Autocorrelation Function (PACF)

### Definition
PACF measures the **correlation between $y_t$ and $y_{t-k}$ after removing the effect of intermediate lags** ($y_{t-1}, y_{t-2}, ..., y_{t-k+1}$).

$$\phi_{kk} = \text{Corr}(y_t, y_{t-k} \mid y_{t-1}, y_{t-2}, ..., y_{t-k+1})$$

### What It Tells You
- The **direct relationship** between current and lag-k values
- The **order of autoregression** — how many past values directly influence the present

### PACF Patterns & Interpretation

| Pattern | Indicates | Action |
|---|---|---|
| **Sharp cutoff after lag p** | AR(p) process | Set `p` = last significant lag |
| **Gradual decay** | MA or ARMA process | Check ACF for MA order |
| **Significant spikes at seasonal lags** | Seasonal AR component | Add seasonal AR terms |

---

## 3. ACF vs. PACF — Side by Side

| Feature | ACF | PACF |
|---|---|---|
| **Measures** | Total correlation with lag-k | Direct correlation with lag-k |
| **Controls for** | Nothing | Intermediate lags (1 to k-1) |
| **Identifies** | MA(q) order | AR(p) order |
| **AR(p) pattern** | Tails off gradually | Cuts off after lag p |
| **MA(q) pattern** | Cuts off after lag q | Tails off gradually |
| **ARMA(p,q) pattern** | Tails off | Tails off |

---

## 4. Identification Rules for ARIMA(p,d,q)

| If ACF shows... | And PACF shows... | Then model is... |
|---|---|---|
| Tails off | Cuts off after lag p | **AR(p)** |
| Cuts off after lag q | Tails off | **MA(q)** |
| Tails off | Tails off | **ARMA(p,q)** — both needed |
| Slow decay (non-stationary) | Slow decay | **Apply differencing first** |

### Step-by-Step Identification

```
Step 1: Plot raw series → check for trend/seasonality
Step 2: Plot ACF of raw series
        ↓
    Slow decay? → Difference the series (d += 1)
    Repeat until ACF shows sudden drop or rapid decay
        ↓
Step 3: On differenced (stationary) series:
        ↓
    PACF cuts off at lag p → AR(p)
    ACF cuts off at lag q → MA(q)
    Both tail off → ARMA(p,q) — try combinations
        ↓
Step 4: Validate with AIC/BIC, check residuals
```

---

## 5. Practical Example

Let me generate ACF/PACF plots for different process types to illustrate:
Now let me create a practical ACF/PACF interpretation guide with a real-world example:
Here is the complete, searchable content on **ACF (Autocorrelation Function)** and **PACF (Partial Autocorrelation Function)** — the two most critical diagnostic tools for ARIMA model identification.

---

## 1. What Are ACF and PACF?

| | **ACF** | **PACF** |
|---|---|---|
| **Full Name** | Autocorrelation Function | Partial Autocorrelation Function |
| **Measures** | Total correlation between $y_t$ and $y_{t-k}$ | Direct correlation between $y_t$ and $y_{t-k}$, removing intermediate effects |
| **Controls for** | Nothing — raw correlation | Lags $1$ through $k-1$ |
| **Primary Use** | Identify MA(q) order | Identify AR(p) order |
| **Key Pattern** | **Cutoff** after lag q → MA(q) | **Cutoff** after lag p → AR(p) |

---

## 2. Mathematical Definitions

### ACF
$$\rho_k = \frac{\text{Cov}(y_t, y_{t-k})}{\text{Var}(y_t)}$$

### PACF
$$\phi_{kk} = \text{Corr}(y_t, y_{t-k} \mid y_{t-1}, y_{t-2}, ..., y_{t-k+1})$$

---

## 3. Identification Rules — The Core Cheat Sheet

| If ACF shows... | And PACF shows... | Then model is... | Parameters |
|---|---|---|---|
| **Tails off gradually** | **Cuts off after lag p** | **AR(p)** | $p$ = last significant PACF lag |
| **Cuts off after lag q** | **Tails off gradually** | **MA(q)** | $q$ = last significant ACF lag |
| **Tails off** | **Tails off** | **ARMA(p,q)** | Both needed — try combinations |
| **Slow decay / high persistence** | **Slow decay** | **Non-stationary** | Apply differencing first ($d \geq 1$) |

### Visual Summary

```
AR(p):    ACF tails off  ~~~>     PACF cuts off |||
MA(q):    ACF cuts off  |||       PACF tails off ~~~>
ARMA:     ACF tails off  ~~~>     PACF tails off ~~~>
Non-stat: ACF slow decay ~~~~~~>   PACF slow decay ~~~~~~>  → DIFFERENCE
```

---

## 4. Reading the Plots — What to Look For

### The Blue Shaded Region
- Represents the **95% confidence interval** (approx. $\pm 2/\sqrt{n}$)
- **Spikes outside this region** are statistically significant
- **Ignore lag 0** — always 1.0 (correlation with itself)

### Significant Spike Interpretation

| Spike Location | Meaning |
|---|---|
| Lag 1 in ACF | Strong immediate autocorrelation |
| Lag 7 in ACF | Weekly pattern (daily data) |
| Lag 12 in ACF | Monthly pattern (monthly data) |
| Lag 365 in ACF | Annual seasonality (daily data) |
| Lag 1 in PACF | Direct AR(1) effect |
| Multiple lags in PACF | Higher-order AR process |

---

## 5. Practical Example: From Non-Stationary to Stationary

Based on the plots generated above:

### **Original Series (Non-Stationary)**
- **ACF**: Very slow decay — correlations stay high across all 40 lags
- **PACF**: Large spike at lag 1, then gradual decay
- **Interpretation**: Clear non-stationarity (trend present)
- **Action**: Apply first differencing ($d = 1$)

### **First Differenced Series (Stationary)**
- **ACF**: Sudden drop — only lag 1 significant, then near zero
- **PACF**: Sudden drop — only lag 1 significant, then near zero
- **Interpretation**: Both tail off quickly → suggests ARMA(1,1) or low-order model
- **Action**: Try ARIMA(1,1,1) or ARIMA(0,1,1) / ARIMA(1,1,0)

---

## 6. Seasonal ACF/PACF Patterns

For seasonal data, look for spikes at **seasonal lags**:

| Data Frequency | Seasonal Lag | Pattern |
|---|---|---|
| Daily | 7, 14, 21... | Weekly seasonality |
| Daily | 365 | Annual seasonality |
| Monthly | 12, 24... | Yearly seasonality |
| Quarterly | 4, 8... | Yearly seasonality |
| Hourly | 24 | Daily seasonality |

**Seasonal ACF/PACF cutoff** indicates seasonal AR(P) or MA(Q) terms for SARIMA.

---

## 7. Step-by-Step ARIMA Identification Workflow

```
Step 1: Plot the raw time series
        ↓ Check for trend, seasonality, variance changes
        ↓
Step 2: Plot ACF of raw series
        ↓
        ACF slow decay? ──Yes──► Apply differencing (d += 1)
        │                         ↓ Repeat Step 2
        └─No──► Continue
        ↓
Step 3: Plot PACF of differenced (stationary) series
        ↓
        PACF cuts off at lag p? ──Yes──► AR(p), set p
        │                                │
        └─No (tails off)──► Check ACF  │
                           ↓            │
        ACF cuts off at lag q? ──Yes──► MA(q), set q
        │                                │
        └─No (tails off)──► ARMA(p,q)──┘
                           Try p=1,2 and q=1,2 combinations
        ↓
Step 4: Fit candidate models, compare AIC/BIC
Step 5: Validate residuals (white noise check)
```

---

## 8. Common ACF/PACF Patterns & Their Meaning

| Pattern | Series Type | ACF Shape | PACF Shape | Model |
|---|---|---|---|---|
| **White noise** | Random, no structure | All lags inside CI | All lags inside CI | None needed |
| **AR(1)** | Mean-reverting | Exponential decay | Spike at lag 1 only | ARIMA(1,0,0) |
| **AR(2)** | Oscillating mean-reversion | Damped sine wave | Spikes at lags 1,2 | ARIMA(2,0,0) |
| **MA(1)** | One-period shock effect | Spike at lag 1 only | Exponential decay | ARIMA(0,0,1) |
| **MA(2)** | Two-period shock effect | Spikes at lags 1,2 | Damped sine wave | ARIMA(0,0,2) |
| **Random walk** | Non-stationary trend | Very slow decay | Spike at lag 1 | ARIMA(0,1,0) |
| **Seasonal AR** | Repeating annual pattern | Spikes at seasonal lags | Spikes at seasonal lags | SARIMA |

---

## 9. Python Code for ACF/PACF

```python
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

# Basic plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(series, ax=axes[0], lags=40, alpha=0.05)
plot_pacf(series, ax=axes[1], lags=40, alpha=0.05)
plt.show()

# On differenced series (for stationarity check)
diff_series = series.diff().dropna()
plot_acf(diff_series, lags=40)
plot_pacf(diff_series, lags=40)
```

---

## 10. Key Takeaways

| Rule | Explanation |
|---|---|
| **ACF identifies MA(q)** | Because MA(q) has memory only of the last q shocks |
| **PACF identifies AR(p)** | Because AR(p) has direct dependence only on last p values |
| **Both tail off → ARMA** | Mixed process needs both components |
| **Slow decay → difference** | Non-stationary series must be differenced first |
| **Always check residuals** | Good model → ACF of residuals = white noise (no significant spikes) |
| **Use AIC/BIC to decide** | When multiple models fit, pick lowest information criterion |

---

## Downloadable Outputs

- [ACF/PACF Pattern Examples](sandbox:///mnt/agents/output/acf_pacf_patterns.png)
- [Practical ACF/PACF Analysis](sandbox:///mnt/agents/output/acf_pacf_practical.png)

## ARIMA(p,d,q) — Complete Parameter Guide

The **(p, d, q)** notation defines the structure of an ARIMA model. Each parameter controls a specific aspect of the time series dynamics.

---

## 1. Parameter Definitions

| Parameter | Component | Meaning | Controls |
|---|---|---|---|
| **p** | **AR** — Autoregressive | Number of lagged observations as predictors | How many past values directly influence the present |
| **d** | **I** — Integrated | Number of differencing operations | How many times to difference to achieve stationarity |
| **q** | **MA** — Moving Average | Number of lagged forecast errors | How many past shocks affect the present |

---

## 2. Detailed Breakdown

### **p — Autoregressive Order**

| Value | Model | Equation Form | When to Use |
|---|---|---|---|
| **p = 0** | No AR component | $y_t = c + \epsilon_t$ | No autocorrelation in stationary series |
| **p = 1** | AR(1) | $y_t = c + \phi_1 y_{t-1} + \epsilon_t$ | Mean-reverting series, one-period momentum |
| **p = 2** | AR(2) | $y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \epsilon_t$ | Oscillating patterns, two-period dynamics |
| **p ≥ 3** | Higher-order AR | More lag terms | Complex cyclical patterns (rarely needed) |

**How to determine p:** Look at **PACF** — last significant lag = p

---

### **d — Differencing Order**

| Value | Operation | Series State | When to Use |
|---|---|---|---|
| **d = 0** | No differencing | Original series | Already stationary (constant mean/variance) |
| **d = 1** | First difference: $\Delta y_t = y_t - y_{t-1}$ | Removes linear trend | Trending data (most common) |
| **d = 2** | Second difference: $\Delta^2 y_t = \Delta(\Delta y_t)$ | Removes quadratic trend | Accelerating/decelerating trends (rare) |
| **d ≥ 3** | Higher-order differencing | Over-differenced | Almost never needed — causes artificial patterns |

**How to determine d:**
- Plot the series — visible trend? → d ≥ 1
- **ADF test**: p < 0.05 → stationary (d = 0), else difference
- **ACF slow decay** → non-stationary, needs differencing

> **Rule of thumb:** Never use d > 2. Over-differencing introduces artificial negative autocorrelation.

---

### **q — Moving Average Order**

| Value | Model | Equation Form | When to Use |
|---|---|---|---|
| **q = 0** | No MA component | $y_t = \mu + \epsilon_t$ | No shock persistence |
| **q = 1** | MA(1) | $y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1}$ | One-period shock effect |
| **q = 2** | MA(2) | $y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2}$ | Two-period shock decay |
| **q ≥ 3** | Higher-order MA | More error lags | Extended shock persistence (rare) |

**How to determine q:** Look at **ACF** — last significant lag = q

---

## 3. Common ARIMA Configurations

| Model | (p,d,q) | Use Case | Example |
|---|---|---|---|
| **White Noise** | (0,0,0) | Random, unpredictable | Fair coin toss |
| **Random Walk** | (0,1,0) | Today's value = yesterday + shock | Stock prices (naive model) |
| **Random Walk with Drift** | (0,1,0) with constant | Trending random walk | GDP, population growth |
| **AR(1)** | (1,0,0) | Mean-reverting, one-period memory | Interest rates |
| **MA(1)** | (0,0,1) | One-period shock smoothing | Manufacturing defects |
| **ARMA(1,1)** | (1,0,1) | Mixed dynamics | Exchange rates |
| **ARIMA(1,1,1)** | (1,1,1) | Trending with mixed dynamics | Most economic series |
| **ARIMA(0,1,1)** | (0,1,1) | Simple exponential smoothing | Sales forecasting |
| **ARIMA(0,2,2)** | (0,2,2) | Holt's linear trend method | Trending series with smoothing |

---

## 4. SARIMA Extension: (p,d,q)(P,D,Q)m

For **seasonal** data, add seasonal parameters:

| Parameter | Meaning | Example |
|---|---|---|
| **P** | Seasonal AR order | Annual pattern in monthly data |
| **D** | Seasonal differencing | Remove yearly trend (e.g., m=12) |
| **Q** | Seasonal MA order | Seasonal shock persistence |
| **m** | Seasonal period | 12 (monthly), 7 (daily), 4 (quarterly) |

**Example:** `SARIMA(1,1,1)(1,1,1)12`
- Regular: AR(1), differenced once, MA(1)
- Seasonal: SAR(1), seasonally differenced, SMA(1), period=12

---

## 5. How to Select (p,d,q) — Decision Tree

```
START
  │
  ├─ Is the series stationary?
  │    ├─ Yes (ADF p < 0.05, ACF drops fast) → d = 0
  │    └─ No (trend visible, ACF slow decay) → d = 1 (or 2)
  │              ↓
  │         Difference d times, re-check ADF
  │              ↓
  ├─ On stationary series, examine PACF:
  │    ├─ Cuts off after lag 1 → p = 1
  │    ├─ Cuts off after lag 2 → p = 2
  │    ├─ Cuts off after lag k → p = k
  │    └─ Tails off gradually → p = 0 (MA process)
  │              ↓
  ├─ On stationary series, examine ACF:
  │    ├─ Cuts off after lag 1 → q = 1
  │    ├─ Cuts off after lag 2 → q = 2
  │    ├─ Cuts off after lag k → q = k
  │    └─ Tails off gradually → q = 0 (AR process)
  │              ↓
  ├─ If both ACF and PACF tail off:
  │    └─ ARMA(p,q) — try p=1,2 and q=1,2 combinations
  │              ↓
  └─ Validate: Compare AIC/BIC, check residual ACF
```

---

## 6. Information Criteria for Model Selection

| Criterion | Formula | Preference | Penalty for Complexity |
|---|---|---|---|
| **AIC** | Akaike Information Criterion | Lower = better | Moderate |
| **AICc** | Corrected AIC (small samples) | Lower = better | Stronger than AIC |
| **BIC** | Bayesian Information Criterion | Lower = better | Strongest (prefers simpler models) |

**Recommendation:** Use **AICc** for small samples (< 100 observations), **BIC** for larger samples or when you prefer parsimony.

---

## 7. Parameter Constraints & Stability

| Parameter | Valid Range | Interpretation |
|---|---|---|
| **AR coefficients ($\phi$)** | $|\phi| < 1$ (stationarity) | $\phi$ close to 1 = high persistence |
| **MA coefficients ($\theta$)** | $|\theta| < 1$ (invertibility) | $\theta$ close to 1 = long shock memory |
| **d** | 0, 1, or 2 | d ≥ 3 causes over-differencing |

**Warning signs of bad model:**
- Coefficients near ±1.0 → near non-stationary/non-invertible
- Residuals still show pattern → wrong (p,d,q) specification
- AIC/BIC worse than simpler model → overfitting

---

## 8. Quick Reference: ACF/PACF → (p,d,q)

| ACF Pattern | PACF Pattern | (p,d,q) | Model Type |
|---|---|---|---|
| Slow decay | Spike at lag 1, then 0 | (1,1,0) | Differenced AR(1) |
| Spike at lag 1, then 0 | Slow decay | (0,1,1) | Differenced MA(1) |
| Slow decay | Slow decay | (1,1,1) | Differenced ARMA(1,1) |
| Sudden drop to 0 | Sudden drop to 0 | (0,0,0) or over-differenced | White noise |
| Seasonal spikes | Seasonal spikes | Add (P,D,Q)m | Seasonal ARIMA |

---

## 9. Python Implementation

```python
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima  # For automatic selection

# Manual ARIMA(2,1,2)
model = ARIMA(series, order=(2, 1, 2))
fitted = model.fit()
print(fitted.summary())

# Auto-ARIMA (automatic p,d,q selection)
auto_model = auto_arima(
    series,
    seasonal=False,
    d=None,           # Let algorithm find d
    max_p=5,
    max_q=5,
    information_criterion='aic',
    trace=True
)
print(auto_model.order)  # Returns best (p,d,q)
```

---

## 10. Summary Table

| Parameter | Identified By | Valid Values | Key Rule |
|---|---|---|---|
| **p** | PACF cutoff | 0, 1, 2, 3... | Last significant PACF lag |
| **d** | ADF test, ACF decay | 0, 1, 2 | Difference until stationary |
| **q** | ACF cutoff | 0, 1, 2, 3... | Last significant ACF lag |

> **Golden Rule:** Start simple. Try (1,1,1) or (0,1,1) or (1,1,0) before complex models. Let AIC/BIC guide final selection./

## Grid Search for ARIMA(p,d,q) — Hyperparameter Tuning

Grid search systematically tests all combinations of (p,d,q) parameters to find the optimal ARIMA model based on a chosen information criterion (AIC, BIC, etc.).

---

## 1. Why Grid Search?

| Challenge | Solution |
|---|---|
| Manual ACF/PACF interpretation is subjective | Grid search is objective and automated |
| Multiple models may seem plausible | Compares all candidates systematically |
| Risk of overfitting with high p,q | Information criteria penalize complexity |
| Seasonal + non-seasonal combinations | Exhaustive search finds best balance |

---

## 2. Grid Search Algorithm

```
Define search space:
    p ∈ {0, 1, 2, ..., p_max}
    d ∈ {0, 1, 2} (rarely > 2)
    q ∈ {0, 1, 2, ..., q_max}

For each combination (p,d,q):
    1. Fit ARIMA(p,d,q) on training data
    2. Check model convergence (successful fit?)
    3. If converged: compute AIC / BIC / AICc
    4. Store results

Select model with lowest criterion value
Validate on hold-out test set
```

---

## 3. Implementation — Manual Grid Search
Here is the complete, searchable content on **Grid Search for ARIMA(p,d,q)** — a systematic approach to hyperparameter tuning.

---

## 1. What Is Grid Search?

Grid search exhaustively tests all combinations of (p,d,q) within a defined search space and selects the optimal model based on an information criterion or out-of-sample performance.

| Approach | Method | Risk |
|---|---|---|
| **Manual (ACF/PACF)** | Subjective interpretation | Human bias, inconsistent |
| **Grid Search** | Automated, objective | Computationally expensive |
| **Auto-ARIMA** | Algorithmic stepwise search | May miss optimal combination |

---

## 2. Search Space Design

| Parameter | Typical Range | Reason |
|---|---|---|
| **p** | 0 to 3 (sometimes 5) | Higher orders rarely needed, risk overfitting |
| **d** | 0, 1, or 2 | d > 2 causes over-differencing |
| **q** | 0 to 3 (sometimes 5) | Higher orders rarely needed |

**Total combinations:** For p∈[0,3], d∈[0,1], q∈[0,3] → **32 models**

> **Rule:** Start small (0–2 for each). Expand only if best model is at boundary.

---

## 3. Selection Criteria

| Criterion | Optimizes | Penalty for Complexity | Best For |
|---|---|---|---|
| **AIC** | In-sample likelihood | Moderate | Balanced model selection |
| **AICc** | In-sample likelihood (small sample corrected) | Strong | Small datasets (< 100 obs) |
| **BIC** | In-sample likelihood | Strongest | Preferring simpler models |
| **Test MAE/RMSE** | Out-of-sample forecast accuracy | None | When forecasting is the goal |
| **Cross-Validation MAE** | Robust out-of-sample performance | None | Best practice for forecasting |

---

## 4. Grid Search Results (from demo)

### Top Models by AIC

| Rank | Model | AIC | BIC | Test MAE | Test MAPE |
|---|---|---|---|---|---|
| 1 | ARIMA(1,1,2) | **8946.78** | 8967.93 | 16.01 | 8.63% |
| 2 | ARIMA(3,0,3) | 8963.43 | 9005.73 | 6.73 | 3.57% |
| 3 | ARIMA(2,0,2) | 8980.17 | 9011.89 | **6.33** | **3.29%** |

### Critical Finding

> **Best AIC ≠ Best Forecast**
>
> ARIMA(1,1,2) has the lowest AIC but the **worst out-of-sample MAE (16.01)**.
> ARIMA(2,0,2) has slightly higher AIC but **3× better forecast accuracy (MAE=6.33)**.

**Why?** AIC optimizes in-sample fit. Differencing (d=1) can amplify noise in long-term forecasts. Modeling trend with AR terms (d=0) often yields more stable predictions.

---

## 5. Recommended Grid Search Workflow

```
Step 1: Define search space
        p ∈ {0, 1, 2, 3}
        d ∈ {0, 1} (or {0, 1, 2})
        q ∈ {0, 1, 2, 3}
        
Step 2: For each (p,d,q):
        - Fit ARIMA on training data
        - Check convergence (skip if fails)
        - Compute AIC, BIC, log-likelihood
        - Generate forecasts on hold-out test set
        - Compute test MAE, RMSE, MAPE
        
Step 3: Rank models by:
        Primary: Test MAE (or CV error) — FORECAST ACCURACY
        Secondary: AIC/BIC — MODEL PARSIMONY
        
Step 4: Select final model:
        - Best test performance within simple models
        - Avoid boundary values (p or q at max) unless justified
        
Step 5: Validate:
        - Residual ACF = white noise?
        - Ljung-Box test p > 0.05?
        - No remaining pattern in residuals?
```

---

## 6. Python Implementation

### Manual Grid Search

```python
import itertools
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error

# Define search space
p_range = range(0, 4)
d_range = range(0, 2)
q_range = range(0, 4)

results = []

for p, d, q in itertools.product(p_range, d_range, q_range):
    try:
        # Fit model
        model = ARIMA(train_data, order=(p, d, q))
        fitted = model.fit()
        
        # Forecast
        forecast = fitted.forecast(steps=len(test_data))
        
        # Evaluate
        mae = mean_absolute_error(test_data, forecast)
        
        results.append({
            'p': p, 'd': d, 'q': q,
            'AIC': fitted.aic,
            'BIC': fitted.bic,
            'MAE': mae,
            'converged': True
        })
    except:
        results.append({
            'p': p, 'd': d, 'q': q,
            'AIC': np.inf, 'BIC': np.inf, 'MAE': np.inf,
            'converged': False
        })

# Select best by test MAE (recommended)
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['MAE'].idxmin()]
print(f"Best model: ARIMA({best['p']},{best['d']},{best['q']})")
```

### Using `pmdarima` (Auto-ARIMA)

```python
from pmdarima import auto_arima

# Automatic grid search with stepwise optimization
model = auto_arima(
    train_data,
    seasonal=False,
    d=None,              # Auto-determine differencing
    max_p=3,
    max_q=3,
    information_criterion='aic',  # or 'bic', 'aicc'
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True        # Faster than full grid search
)

print(model.order)  # Best (p,d,q)
```

### Using `statsforecast` (Fast, Scalable)

```python
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

sf = StatsForecast(
    models=[AutoARIMA(season_length=7, max_p=3, max_q=3)],
    freq='D'
)

sf.fit(df)
```

---

## 7. Grid Search Best Practices

| Practice | Why It Matters |
|---|---|
| **Use train/test split** | Prevents overfitting to in-sample criteria |
| **Prefer test MAE over AIC** | AIC optimizes fit, not forecast accuracy |
| **Check convergence** | Some combinations fail to converge — exclude them |
| **Avoid d > 2** | Over-differencing creates artificial patterns |
| **Start with small p,q** | Higher orders rarely improve forecasts |
| **Validate residuals** | Best model should leave white noise residuals |
| **Consider BIC for parsimony** | Stronger penalty avoids overfitting |
| **Use cross-validation** | More robust than single train/test split |

---

## 8. Common Pitfalls

| Pitfall | Problem | Solution |
|---|---|---|
| **Optimizing AIC only** | Best in-sample fit, poor forecasts | Always check out-of-sample MAE/RMSE |
| **Large search spaces** | Slow, overfitting risk | Keep p,q ≤ 3 initially |
| **Ignoring convergence** | Failed fits bias results | Filter out non-converged models |
| **Not validating residuals** | Missed model misspecification | Ljung-Box test on residuals |
| **Using d=2 unnecessarily** | Over-differencing | Check if d=1 is sufficient |

---

## 9. Summary: Grid Search Decision Framework

```
┌─────────────────────────────────────────┐
│  1. Define p∈[0,3], d∈[0,1], q∈[0,3]  │
│  2. Fit all combinations on training set │
│  3. Compute AIC/BIC + test MAE for each │
│  4. Filter: converged models only      │
│  5. Rank by test MAE (primary)          │
│  6. Break ties with AIC/BIC (secondary) │
│  7. Validate residuals = white noise     │
│  8. Final model selected                │
└─────────────────────────────────────────┘
```

---

## 10. Key Takeaways

| Insight | Action |
|---|---|
| **AIC is not enough** | Always verify with out-of-sample error |
| **Simpler models often win** | ARIMA(2,0,2) beat ARIMA(3,0,3) in our demo |
| **d=0 can outperform d=1** | AR terms can capture trend without differencing noise |
| **Boundary values = warning** | If best p or q is at max, expand search cautiously |
| **Cross-validation > single split** | Rolling-origin CV is gold standard for time series |

---

## Downloadable Output

- [Grid Search Analysis Visualization](sandbox:///mnt/agents/output/grid_search_analysis.png)

# **facebook prophet**

## Facebook Prophet

Prophet is an open-source forecasting tool developed by Facebook (Meta) designed for **business time series** with strong seasonal effects and multiple seasonality. It uses an **additive regression model** rather than traditional ARIMA.

---

## 1. Core Philosophy

| Traditional ARIMA | Prophet |
|---|---|
| Assumes linear/autoregressive structure | Assumes **piecewise linear trend + seasonality** |
| Requires stationarity | Handles **non-stationary data naturally** |
| Manual parameter tuning (p,d,q) | **Automatic** — minimal configuration |
| Struggles with missing data, outliers | **Robust** to missing data, outliers, trend changes |
| Single seasonality | **Multiple seasonalities** (daily, weekly, yearly) |
| No built-in holidays | **Holiday effects** built-in |

---

## 2. Model Equation

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

| Component | Symbol | Description |
|---|---|---|
| **Trend** | $g(t)$ | Piecewise linear or logistic growth with changepoints |
| **Seasonality** | $s(t)$ | Fourier series representing periodic patterns |
| **Holidays** | $h(t)$ | User-provided list of holidays with effects |
| **Error** | $\epsilon_t$ | Normally distributed noise |

---

## 3. Components in Detail

### Trend $g(t)$ — Two Options

| Type | Formula | Use Case |
|---|---|---|
| **Linear** | $g(t) = (k + a(t)^T \delta) \cdot t + (m + a(t)^T \gamma)$ | Constant growth rate with changepoints |
| **Logistic** | $g(t) = \frac{C}{1 + \exp(-(k + a(t)^T \delta)(t - (m + a(t)^T \gamma))}$ | Growth with carrying capacity $C$ |

**Changepoints:** Prophet automatically detects **trend change points** where the growth rate shifts. You can also specify them manually.

---

### Seasonality $s(t)$ — Fourier Series

$$s(t) = \sum_{n=1}^{N} \left( a_n \cos\left(\frac{2\pi n t}{P}\right) + b_n \sin\left(\frac{2\pi n t}{P}\right) \right)$$

| Parameter | Default | Description |
|---|---|---|
| **yearly_seasonality** | True | Annual cycle (P=365.25 days) |
| **weekly_seasonality** | True | Weekly cycle (P=7 days) |
| **daily_seasonality** | False | Intra-day cycle (P=1 day) |
| **N (Fourier terms)** | 10 (yearly), 3 (weekly) | Higher = more flexible seasonality |

---

### Holidays $h(t)$

```python
# Custom holidays with effects
holidays = pd.DataFrame({
    'holiday': 'black_friday',
    'ds': pd.to_datetime(['2024-11-29', '2025-11-28']),
    'lower_window': -1,   # Effect starts 1 day before
    'upper_window': 1,    # Effect ends 1 day after
})
```

---

## 4. Prophet vs. ARIMA Comparison

| Feature | Prophet | ARIMA/SARIMA |
|---|---|---|
| **Ease of use** | ⭐ Very easy — minimal tuning | Requires expertise (p,d,q,S) |
| **Trend handling** | Automatic changepoint detection | Manual differencing |
| **Multiple seasonality** | Built-in (yearly + weekly + daily) | SARIMA limited to one seasonality |
| **Holidays** | Native support | Must engineer manually |
| **Missing data** | Handles naturally | Requires imputation |
| **Outliers** | Robust (uses Bayesian regression) | Sensitive |
| **Uncertainty intervals** | Built-in (MCMC or MAP) | Requires manual computation |
| **Interpretability** | High — decomposable components | Medium — coefficient-based |
| **Small data (< 100 obs)** | Works well | ARIMA may be better |
| **Long-term forecasts** | Trend uncertainty grows | Can be more stable |
| **Exogenous variables** | Limited (regressors) | Full ARIMAX support |

---

## 5. When to Use Prophet

| Use Prophet | Don't Use Prophet |
|---|---|
| Daily/weekly business data with clear seasonality | High-frequency data (hourly/minutely) — use ARIMA or deep learning |
| Multiple seasonal patterns (yearly + weekly) | Pure autoregressive dynamics (use ARIMA) |
| Need automatic trend changepoint detection | Need full exogenous variable control (use ARIMAX) |
| Missing data, outliers common | Very small datasets (< 20 observations) |
| Need interpretable decomposition | Need state-space or transfer function models |
| Forecasting with holiday effects | Real-time streaming forecasts |

---

## 6. Practical Implementation
Prophet installation is having issues in this environment. Let me provide the complete searchable content on Facebook Prophet based on the established framework and theory, along with a manual implementation of Prophet's core concepts.

---

## Facebook Prophet — Complete Guide

---

## 1. What Is Prophet?

**Prophet** is an open-source forecasting procedure developed by Facebook (Meta) in 2017. It is designed for **business time series** that exhibit:

- Strong **seasonal effects**
- **Multiple seasonality** (yearly, weekly, daily)
- **Trend changes** (growth, flat, declining)
- **Holiday effects**
- **Missing data** and **outliers**

Unlike ARIMA, Prophet uses an **additive regression model** with a piecewise linear trend and Fourier series seasonality.

---

## 2. Model Equation

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

| Component | Symbol | Description | Formula |
|---|---|---|---|
| **Trend** | $g(t)$ | Piecewise linear or logistic growth | $g(t) = (k + a(t)^T\delta) \cdot t + (m + a(t)^T\gamma)$ |
| **Seasonality** | $s(t)$ | Fourier series for periodic patterns | $\sum_{n=1}^{N} \left(a_n \cos\frac{2\pi nt}{P} + b_n \sin\frac{2\pi nt}{P}\right)$ |
| **Holidays** | $h(t)$ | User-defined holiday effects | $\sum_{i} \kappa_i \cdot \mathbf{1}_{t \in D_i}$ |
| **Error** | $\epsilon_t$ | Normally distributed noise | $\epsilon_t \sim \mathcal{N}(0, \sigma^2)$ |

---

## 3. Components in Detail

### 3.1 Trend $g(t)$ — Two Types

| Type | Formula | Use Case |
|---|---|---|
| **Linear** | $g(t) = (k + a(t)^T\delta) \cdot t + (m + a(t)^T\gamma)$ | Constant growth with changepoints |
| **Logistic** | $g(t) = \frac{C}{1 + \exp(-(k + a(t)^T\delta)(t - (m + a(t)^T\gamma)))}$ | Growth with saturation/capacity $C$ |

**Changepoints:**
- Automatically detected at **25 default points** evenly spaced in first 80% of time series
- Trend slope can change at each changepoint
- **Changepoint prior scale** $\tau$ controls flexibility:
  - $\tau = 0.001$ → Rigid, almost no changepoints
  - $\tau = 0.05$ (default) → Moderate flexibility
  - $\tau = 0.5$ → Very flexible, many trend changes

---

### 3.2 Seasonality $s(t)$ — Fourier Series

Prophet represents seasonality using **Fourier series** rather than dummy variables:

$$s(t) = \sum_{n=1}^{N} \left( a_n \cos\left(\frac{2\pi n t}{P}\right) + b_n \sin\left(\frac{2\pi n t}{P}\right) \right)$$

| Seasonality | Period $P$ | Default Fourier Terms $N$ | Enable |
|---|---|---|---|
| **Yearly** | 365.25 days | 10 | `yearly_seasonality=True` |
| **Weekly** | 7 days | 3 | `weekly_seasonality=True` |
| **Daily** | 1 day | 4 | `daily_seasonality=False` |

Higher $N$ = more flexible, risk of overfitting.

---

### 3.3 Holidays $h(t)$

Custom holidays with effects:

```python
holidays = pd.DataFrame({
    'holiday': 'black_friday',
    'ds': pd.to_datetime(['2024-11-29', '2025-11-28']),
    'lower_window': -1,   # Effect starts 1 day before
    'upper_window': 1,    # Effect ends 1 day after
})
```

---

## 4. Prophet vs. ARIMA — Head-to-Head

| Feature | Prophet | ARIMA/SARIMA |
|---|---|---|
| **Ease of use** | ⭐ Very easy — minimal tuning | Requires expertise (p,d,q,S) |
| **Trend** | Automatic changepoint detection | Manual differencing (d) |
| **Multiple seasonality** | Built-in (yearly + weekly + daily) | Limited to one seasonality |
| **Holidays** | Native support | Must engineer manually |
| **Missing data** | Handles naturally | Requires imputation |
| **Outliers** | Robust (Bayesian regression) | Sensitive |
| **Uncertainty** | Built-in MCMC/MAP intervals | Manual computation |
| **Interpretability** | High — decomposable | Medium — coefficients |
| **Small data** | Works well (< 100 obs) | May overfit |
| **Exogenous vars** | Limited (regressors) | Full ARIMAX support |
| **Speed** | Fast | Very fast |
| **Long-term forecasts** | Trend uncertainty grows | Can be more stable |

---

## 5. When to Use Prophet

| ✅ Use Prophet | ❌ Don't Use Prophet |
|---|---|
| Daily/weekly business data with seasonality | High-frequency (hourly/minutely) — use ARIMA or deep learning |
| Multiple seasonal patterns | Pure autoregressive dynamics |
| Need automatic trend changepoints | Need full exogenous control (use ARIMAX) |
| Missing data, outliers common | Very small datasets (< 20 obs) |
| Need interpretable decomposition | Need state-space models |
| Forecasting with holidays | Real-time streaming |
| Non-technical users need to run forecasts | Need transfer function models |

---

## 6. Prophet Hyperparameters

| Parameter | Default | Range | Effect |
|---|---|---|---|
| `changepoint_prior_scale` | 0.05 | 0.001 – 0.5 | Trend flexibility (higher = more changepoints) |
| `seasonality_prior_scale` | 10.0 | 0.01 – 10 | Seasonality flexibility |
| `holidays_prior_scale` | 10.0 | 0.01 – 10 | Holiday effect flexibility |
| `n_changepoints` | 25 | 15 – 50 | Number of potential changepoints |
| `changepoint_range` | 0.8 | 0.8 – 0.95 | % of data where changepoints allowed |
| `yearly_seasonality` | 'auto' | True/False/'auto' | Enable yearly Fourier series |
| `weekly_seasonality` | 'auto' | True/False/'auto' | Enable weekly Fourier series |
| `daily_seasonality` | 'auto' | True/False/'auto' | Enable daily Fourier series |
| `interval_width` | 0.80 | 0.5 – 0.99 | Prediction interval width |
| `growth` | 'linear' | 'linear'/'logistic' | Trend model type |

---

## 7. Prophet Code Template

### Basic Usage

```python
from prophet import Prophet
import pandas as pd

# Data must have 'ds' (datetime) and 'y' (value) columns
df = pd.DataFrame({
    'ds': pd.date_range('2020-01-01', periods=1000, freq='D'),
    'y': your_time_series_values
})

# Initialize and fit
m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=0.05
)
m.fit(df)

# Create future dataframe
future = m.make_future_dataframe(periods=365)  # Forecast 1 year ahead

# Predict
forecast = m.predict(future)

# Plot
fig1 = m.plot(forecast)           # Forecast with uncertainty intervals
fig2 = m.plot_components(forecast)  # Trend + seasonality decomposition
```

### With Custom Holidays

```python
# Define custom holidays
holidays = pd.DataFrame({
    'holiday': 'super_bowl',
    'ds': pd.to_datetime(['2024-02-11', '2025-02-09']),
    'lower_window': -2,
    'upper_window': 1,
})

m = Prophet(holidays=holidays)
m.fit(df)
```

### With Additive Regressors (Exogenous Variables)

```python
# Add external regressors
df['temperature'] = temperature_data  # Must be in training AND future data

m = Prophet()
m.add_regressor('temperature')
m.fit(df)

# Future dataframe must include the regressor
future = m.make_future_dataframe(periods=30)
future['temperature'] = future_temperature_values
forecast = m.predict(future)
```

### Cross-Validation

```python
from prophet.diagnostics import cross_validation, performance_metrics

# Rolling cross-validation
df_cv = cross_validation(
    m,
    initial='730 days',      # First training period
    period='180 days',       # Spacing between cutoffs
    horizon='365 days'       # Forecast horizon
)

# Performance metrics
df_p = performance_metrics(df_cv)
print(df_p[['horizon', 'mae', 'rmse', 'mape']].head())
```

---

## 8. Prophet Diagnostics

| Method | Purpose | Code |
|---|---|---|
| **Plot forecast** | Visualize predictions + intervals | `m.plot(forecast)` |
| **Plot components** | Decompose trend + seasonality + holidays | `m.plot_components(forecast)` |
| **Cross-validation** | Rolling-origin validation | `prophet.diagnostics.cross_validation` |
| **Performance metrics** | MAE, RMSE, MAPE by horizon | `prophet.diagnostics.performance_metrics` |
| **Changepoint visualization** | See where trend changed | `m.plot(forecast); add_changepoints_to_plot` |

---

## 9. Grid Search for Prophet

Unlike ARIMA, Prophet has fewer hyperparameters to tune:

```python
import itertools
from sklearn.metrics import mean_absolute_error

# Define search space
param_grid = {
    'changepoint_prior_scale': [0.001, 0.01, 0.05, 0.1, 0.5],
    'seasonality_prior_scale': [0.01, 0.1, 1.0, 10.0],
    'n_changepoints': [15, 25, 50]
}

best_mae = float('inf')
best_params = None

for cps, sps, nc in itertools.product(
    param_grid['changepoint_prior_scale'],
    param_grid['seasonality_prior_scale'],
    param_grid['n_changepoints']
):
    m = Prophet(
        changepoint_prior_scale=cps,
        seasonality_prior_scale=sps,
        n_changepoints=nc
    )
    m.fit(train)
    
    future = m.make_future_dataframe(periods=len(test))
    forecast = m.predict(future)
    preds = forecast.iloc[-len(test):]['yhat']
    
    mae = mean_absolute_error(test['y'], preds)
    if mae < best_mae:
        best_mae = mae
        best_params = {'cps': cps, 'sps': sps, 'nc': nc}

print(f"Best params: {best_params}, MAE: {best_mae}")
```

---

## 10. Prophet Limitations

| Limitation | Explanation | Workaround |
|---|---|---|
| **No autoregressive component** | Doesn't model $y_{t-1} \to y_t$ directly | Use ARIMA for AR dynamics |
| **Limited exogenous support** | Only additive regressors | Use ARIMAX or ML models |
| **Trend uncertainty grows fast** | Long-term forecasts very wide | Use ensemble methods |
| **Assumes Gaussian noise** | Not robust to heavy tails | Transform data (log, Box-Cox) |
| **Struggles with abrupt regime changes** | Smooth trend transitions | Manual changepoints |
| **Not designed for high frequency** | Best for daily/weekly | Aggregate or use ARIMA |

---

## 11. Prophet vs. Other Models — Expected Performance

Based on research and typical benchmarks:

| Model | Typical MAPE (daily retail) | Strength |
|---|---|---|
| **Naive** | 20–30% | Baseline |
| **ARIMA** | 15–25% | Simple trends |
| **Holt-Winters** | 10–20% | Single seasonality |
| **Prophet** | **8–15%** | Multiple seasonality + holidays |
| **XGBoost + lags** | 10–18% | Many exogenous features |
| **Deep learning (LSTM)** | 8–15% | Large, complex datasets |

---

## 12. Key Takeaways

| Rule | Rationale |
|---|---|
| **Start with defaults** | Prophet is designed to work out-of-the-box |
| **Tune `changepoint_prior_scale` first** | Most impactful parameter |
| **Use cross-validation** | Prophet's built-in CV handles time series correctly |
| **Add holidays for business data** | Often the biggest accuracy gain |
| **Check component plots** | Ensures seasonality/trend are reasonable |
| **Don't overfit seasonality** | Too many Fourier terms = noise fitting |
| **For high frequency, aggregate first** | Prophet works best at daily/weekly granularity |

---

## 13. Quick Reference: Prophet Workflow

```
1. Prepare data → columns 'ds' (datetime) and 'y' (value)
         ↓
2. Initialize Prophet(
       yearly_seasonality=True,
       weekly_seasonality=True,
       changepoint_prior_scale=0.05
   )
         ↓
3. Add holidays / regressors (optional)
         ↓
4. m.fit(df)
         ↓
5. future = m.make_future_dataframe(periods=N)
         ↓
6. forecast = m.predict(future)
         ↓
7. Evaluate with cross_validation()
         ↓
8. Tune hyperparameters via grid search
         ↓
9. Final model → deploy
```

---

## 14. Prophet Ecosystem

| Tool | Purpose |
|---|---|
| `prophet` | Core forecasting library |
| `prophet.diagnostics` | Cross-validation, performance metrics |
| `prophet.plot` | Visualization utilities |
| `neuralprophet` | Neural network extension of Prophet |
| `prophetverse` | Bayesian variant with more flexibility |

## Prophet: Univariate vs. Multivariate

---

## 1. Prophet as a Univariate Model (Default)

By default, Prophet is **univariate** — it uses only the target variable's history (`ds`, `y`) plus internal components (trend, seasonality, holidays).

### Univariate Equation

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

| Component | Source | Description |
|---|---|---|
| $g(t)$ | Internal | Piecewise linear/logistic trend |
| $s(t)$ | Internal | Fourier series seasonality (yearly, weekly, daily) |
| $h(t)$ | Internal | Built-in or custom holidays |
| $\epsilon_t$ | Internal | Observation noise |

### Default Univariate Code

```python
from prophet import Prophet

# Only ds (datetime) and y (value) required
df = pd.DataFrame({
    'ds': pd.date_range('2020-01-01', periods=1000, freq='D'),
    'y': sales_data  # Single target variable
})

m = Prophet()  # Default: univariate
m.fit(df)

future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)
```

**Prophet automatically generates:**
- Trend changepoints
- Yearly seasonality (Fourier terms)
- Weekly seasonality (Fourier terms)
- Holiday effects (if holidays provided)

---

## 2. Prophet as a Multivariate Model (With Regressors)

Prophet becomes **multivariate** when you add **external regressors** — exogenous variables that influence the target.

### Multivariate Equation

$$y(t) = g(t) + s(t) + h(t) + \sum_{i} \beta_i x_i(t) + \epsilon_t$$

| Component | Source | Type |
|---|---|---|
| $g(t)$ | Internal | Trend |
| $s(t)$ | Internal | Seasonality |
| $h(t)$ | Internal | Holidays |
| $\sum \beta_i x_i(t)$ | **External** | **Added regressors** |
| $\epsilon_t$ | Internal | Noise |

### Types of Regressors in Prophet

| Regressor Type | Function | Use Case |
|---|---|---|
| `add_regressor()` | Standard linear effect | Temperature, price, promotion |
| `add_seasonality()` | Custom periodic pattern | Custom business cycles |
| `add_country_holidays()` | Built-in holiday calendars | US, UK, India holidays |

---

## 3. Adding Regressors: Code Examples

### Standard Additive Regressor

```python
from prophet import Prophet

# Data with exogenous variables
df = pd.DataFrame({
    'ds': dates,
    'y': sales,
    'temperature': temp_values,      # External feature
    'promotion': promo_flag,         # Binary feature
    'price': price_values            # Continuous feature
})

m = Prophet()

# Add regressors BEFORE fitting
m.add_regressor('temperature')
m.add_regressor('promotion')
m.add_regressor('price')

m.fit(df)

# Future dataframe MUST include regressors
future = m.make_future_dataframe(periods=30)
future['temperature'] = future_temp_values
future['promotion'] = future_promo_flags
future['price'] = future_prices

forecast = m.predict(future)
```

### Regressor with Prior Scale (Regularization)

```python
# Strong prior = less flexible coefficient
m.add_regressor('temperature', prior_scale=0.1)

# Weak prior = more flexible coefficient  
m.add_regressor('promotion', prior_scale=10.0)

# Standardize the regressor
m.add_regressor('price', standardize=True)
```

| Parameter | Default | Effect |
|---|---|---|
| `prior_scale` | 10.0 | Lower = stronger regularization (coefficient closer to 0) |
| `standardize` | 'auto' | Scales regressor to mean 0, std 1 |
| `mode` | 'additive' | 'additive' or 'multiplicative' effect |

---

## 4. Prophet Multivariate: Key Limitations

| Limitation | Explanation | Workaround |
|---|---|---|
| **Only additive regressors** | Effects sum linearly | Transform features (log, interactions) before adding |
| **No lagged regressors natively** | Can't auto-use $x_{t-1}$ | Create lag features manually in dataframe |
| **No endogenous multivariate** | Can't predict multiple targets simultaneously | Run separate Prophet models or use VAR |
| **Regressors must be known for future** | Need future values of all regressors | Forecast regressors first, or use scenario analysis |
| **No autoregressive component** | Doesn't use $y_{t-1}$ directly | ARIMA handles this better |
| **No cross-series learning** | Each series modeled independently | Use DeepAR or TFT for multiple related series |

---

## 5. Prophet vs. True Multivariate Models

| Feature | Prophet + Regressors | VAR | ARIMAX | XGBoost | DeepAR |
|---|---|---|---|---|---|
| **Multiple targets** | ❌ No | ✅ Yes | ❌ No | ❌ No | ✅ Yes |
| **Exogenous variables** | ✅ Yes | ❌ No | ✅ Yes | ✅ Yes | ✅ Yes |
| **Endogenous system** | ❌ No | ✅ Yes | ❌ No | ❌ No | ✅ Yes |
| **Lag effects** | Manual | Built-in | Built-in | Manual | Built-in |
| **Non-linear effects** | ❌ Linear only | ❌ Linear | ❌ Linear | ✅ Yes | ✅ Yes |
| **Probabilistic** | ✅ Yes | ❌ No | ❌ No | ❌ No | ✅ Yes |
| **Interpretability** | ✅ High | ✅ High | ✅ High | ⚠️ Medium | ⚠️ Low |
| **Multiple seasonality** | ✅ Yes | ❌ No | ❌ No | Manual | ✅ Yes |

---

## 6. When to Use Prophet: Univariate vs. Multivariate

### Use **Univariate Prophet** when:
- Only sales/history data available
- External factors unknown or unreliable
- Need quick baseline forecast
- Strong seasonal patterns dominate
- Explaining seasonality/trend is priority

### Use **Multivariate Prophet** when:
- Known drivers exist (weather, promotions, price)
- Future regressors are predictable/known
- Need to quantify driver impact ("how much did promotion boost sales?")
- Simple linear effect of regressors is sufficient

### **Don't use Prophet** when:
- Need to predict multiple targets together → Use **VAR** or **DeepAR**
- Complex non-linear interactions → Use **XGBoost** or **LightGBM**
- Lagged target effects critical → Use **ARIMA** or **LSTM**
- Regressors unknown for future → Use **univariate** or forecast regressors first

---

## 7. Practical Workflow: Univariate → Multivariate

```
Step 1: Start Univariate
        m = Prophet()
        m.fit(df[['ds', 'y']])
        baseline = evaluate(forecast)
        
Step 2: Identify External Drivers
        - Correlation analysis
        - Domain knowledge (promotions, weather, price)
        
Step 3: Add Regressors
        m.add_regressor('temperature')
        m.add_regressor('promotion')
        m.fit(df[['ds', 'y', 'temperature', 'promotion']])
        
Step 4: Evaluate Improvement
        - Did MAE/MAPE decrease?
        - Are coefficients reasonable?
        
Step 5: Handle Future Regressors
        Option A: Known future values (calendar, planned promotions)
        Option B: Forecast regressors separately
        Option C: Scenario analysis (what-if)
```

---

## 8. Complete Multivariate Prophet Example

```python
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

# --- DATA PREPARATION ---
df = pd.DataFrame({
    'ds': pd.date_range('2020-01-01', periods=1462, freq='D'),
    'y': sales_values,
    'temperature': temp_values,           # Continuous exogenous
    'promotion': promo_binary,           # Binary: 1 = promotion day
    'price': price_values,               # Competitor price index
    'day_of_month': df['ds'].dt.day      # Can also add time features
})

# Split train/test
train = df.iloc[:-365]
test = df.iloc[-365:]

# --- MODEL SETUP ---
m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=0.05
)

# Add regressors with different priors
m.add_regressor('temperature', prior_scale=10.0)    # Flexible
m.add_regressor('promotion', prior_scale=1.0)       # Moderate
m.add_regressor('price', prior_scale=5.0)         # Balanced

# --- FIT ---
m.fit(train)

# --- FORECAST (need future regressors) ---
future = m.make_future_dataframe(periods=365)

# Must provide future values for ALL regressors
future['temperature'] = future_temp_forecast
future['promotion'] = future_promo_calendar
future['price'] = future_price_assumption

forecast = m.predict(future)

# --- EVALUATION ---
preds = forecast.iloc[-365:]['yhat']
mae = mean_absolute_error(test['y'], preds)

# --- INTERPRETATION ---
# Regressor coefficients
print(m.params['beta'])  # β coefficients for each regressor

# Component plot shows regressor contribution
fig = m.plot_components(forecast)
```

---

## 9. Regressor Impact Analysis

```python
# Extract regressor coefficients
from prophet.utilities import regressor_coefficients

coef = regressor_coefficients(m)
print(coef[['regressor', 'coef_lower', 'coef', 'coef_upper']])

# Example output:
#    regressor  coef_lower   coef  coef_upper
# 0  temperature      0.15   0.23        0.31   ← +0.23 units per degree
# 1    promotion      45.2   52.8        60.4   ← +52.8 units on promo days
# 2        price      -1.8   -1.2        -0.6   ← -1.2 units per price index
```

---

## 10. Summary: Prophet's Multivariate Capability

| Aspect | Prophet Capability | Limitation |
|---|---|---|
| **Exogenous variables** | ✅ Additive regressors | Linear only, no interactions |
| **Lagged effects** | ⚠️ Manual creation | No built-in lag handling |
| **Multiple targets** | ❌ Not supported | One target per model |
| **Endogenous systems** | ❌ Not supported | Use VAR instead |
| **Future regressors** | ✅ Required | Must know or forecast them |
| **Regularization** | ✅ Prior scales | Less flexible than ML |
| **Interpretability** | ✅ Excellent | Clear coefficient estimates |

---

## 11. Decision: Prophet vs. Alternatives for Multivariate

| Scenario | Recommendation |
|---|---|
| Single target + known regressors + need interpretability | **Prophet + regressors** |
| Single target + many features + non-linear | **XGBoost / LightGBM** |
| Single target + AR dynamics important | **ARIMAX** |
| Multiple targets, system of equations | **VAR / VARMAX** |
| Multiple targets + probabilistic + large data | **DeepAR / TFT** |
| Hierarchical data (total → regions → stores) | **Prophet (separate) + reconciliation** or **Hierarchical forecasting** |